# Exercise 3: Neural posterior estimation with `sbi`

**Goal.** Estimate the same posterior with neural posterior estimation (NPE) from the `sbi` package, then compare rejection ABC, the MDN and NPE side by side.

NPE is the same idea as Exercise 2: train a conditional density estimator $q_\phi(\theta \mid x)$ by maximum likelihood on simulated pairs. The differences are a more flexible density estimator (a normalizing flow) and `sbi`'s built-in training, validation and sampling. We simulate by hand rather than with `sbi`'s helper functions, so every step of simulate → train → sample stays visible.

In [1]:
import math

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import nn

from sbi.inference import NPE
from sbi.utils import BoxUniform

/opt/miniconda3/envs/road2sbi/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## The simulator and the observation

This cell is **identical in Exercises 1, 2 and 3**, so results are comparable across notebooks. Feel free to change the constants, but change them in all three notebooks.

- **Prior:** $\theta \sim \mathcal{U}(\theta_{low}, \theta_{high})$.
- **Simulator:** $x = A\sin(\omega\,\theta) + b\,\theta + \varepsilon$, with $\varepsilon \sim \mathcal{N}\big(0, \sigma(\theta)^2\big)$ and $\sigma(\theta) = 1.5\,(\sin\theta + 1.5)$. The noise level itself depends on $\theta$: the noise is *heteroscedastic*.
- **Observation:** `x_obs` is one simulation at `THETA_TRUE`, generated with its own fixed seed so it is the same in every notebook.
- **Reference posterior:** `grid_posterior` computes the exact posterior numerically on a grid. That is only possible because we wrote the simulator and know its likelihood; in real SBI problems you can't. We use it purely as a reference. Set `SHOW_GROUND_TRUTH = False` to hide it from the plots.

In [2]:
# ---------------- Shared setup: identical in exercises 1, 2 and 3 ----------------
SEED = 42
torch.manual_seed(SEED)

# Prior: theta ~ Uniform(THETA_LOW, THETA_HIGH), as a PyTorch distribution -- the same object `sbi` needs in Exercise 3
THETA_LOW, THETA_HIGH = -10.5, 10.5
prior = BoxUniform(low=torch.tensor([THETA_LOW]), high=torch.tensor([THETA_HIGH]))

# Simulator: x = A sin(OMEGA theta) + SLOPE theta + noise, with a theta-dependent noise level
A, OMEGA, SLOPE = 7.0, 0.75, 1.0

def mean_x(theta):
    return A * torch.sin(OMEGA * theta) + SLOPE * theta

def noise_std(theta):
    return 1.5 * (torch.sin(theta) + 1.5)

def simulate(theta):
    """theta: a tensor of any shape. Returns x of the same shape, drawn from the current torch RNG state."""
    theta = torch.as_tensor(theta, dtype=torch.float32)
    return mean_x(theta) + noise_std(theta) * torch.randn_like(theta)

# The observation we do inference on
THETA_TRUE = -1.0
torch.manual_seed(SEED)
x_obs = simulate(torch.tensor(THETA_TRUE)).item()

# Numerical ground-truth posterior on a grid (possible only because we know the likelihood)
SHOW_GROUND_TRUTH = True

def grid_posterior(x, n_grid=2001):
    theta_grid = torch.linspace(THETA_LOW, THETA_HIGH, n_grid)
    log_lik = -0.5 * ((x - mean_x(theta_grid)) / noise_std(theta_grid)) ** 2 - torch.log(noise_std(theta_grid))
    density = torch.exp(log_lik - log_lik.max())
    density = density / (density.sum() * (theta_grid[1] - theta_grid[0]))
    return theta_grid.numpy(), density.numpy()

# Plot colors, consistent across notebooks
COLORS = {"abc": "#E69F00", "mdn": "#009E73", "npe": "#0072B2", "truth": "0.3", "prior": "0.75"}

## Prior

We already defined `prior` as a PyTorch distribution in the shared setup above -- that's exactly what `sbi` needs. `BoxUniform` is a uniform distribution over a box, here in 1D.

In [3]:
print(prior.sample((3,)))

tensor([[-2.3006],
        [ 2.1188],
        [-5.1120]])


## Simulating the training data

We call our NumPy simulator directly. `sbi` expects float32 tensors of shape `(n_simulations, n_dimensions)`. Even with 1D $\theta$ and $x$, that trailing dimension of size 1 is required, and the observation needs the same format.

In [4]:
N_TRAIN = 5000

theta_train = prior.sample((N_TRAIN,))
x_train = simulate(theta_train)
x_obs_t = torch.tensor([[x_obs]], dtype=torch.float32)
print(f"theta_train {tuple(theta_train.shape)}, x_train {tuple(x_train.shape)}, x_obs_t {tuple(x_obs_t.shape)}")

theta_train (5000, 1), x_train (5000, 1), x_obs_t (1, 1)


## Training NPE

Training takes three calls:

1. `inference = NPE(prior=prior, density_estimator="nsf")`. `"nsf"` is a neural spline flow.
2. `inference.append_simulations(theta, x)` hands over the simulated pairs.
3. `inference.train()` trains by maximum likelihood. It holds out 10% of the data for validation and stops once the validation loss stops improving.

Then `inference.build_posterior()` wraps the trained density estimator in a posterior object you can sample from.

**Exercise 3a.** Train NPE on `theta_train` and `x_train`, and build the posterior.

In [ ]:
torch.manual_seed(SEED)

# EXERCISE 3a: create an NPE object with our prior and density_estimator="nsf", append (theta_train, x_train),
# train it, and store the result of build_posterior() in `posterior`.
inference = ...
...  # inference.append_simulations(theta_train, x_train)
...  # inference.train()
posterior = ...

print(posterior)

**Exercise 3b.** Draw 10,000 samples from the posterior at $x_{obs}$ with `posterior.sample((n,), x=x_obs_t)`, and convert them to a 1D numpy array called `npe_posterior_samples`.

In [ ]:
# EXERCISE 3b: sample 10_000 thetas given x_obs_t, then .numpy().ravel() them into `npe_posterior_samples`.
npe_posterior_samples = ...

assert npe_posterior_samples.shape == (10_000,), "expected a 1D array of 10,000 samples"
assert np.all((npe_posterior_samples >= THETA_LOW) & (npe_posterior_samples <= THETA_HIGH)), "samples should lie inside the prior"

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(npe_posterior_samples, bins=80, range=(THETA_LOW, THETA_HIGH), density=True,
        color=COLORS["npe"], alpha=0.6, label="NPE posterior")
if SHOW_GROUND_TRUTH:
    ax.plot(*grid_posterior(x_obs), color=COLORS["truth"], lw=1.5, label="true posterior (grid)")
ax.axvline(THETA_TRUE, color="k", ls="--", label=r"$\theta_{true}$")
ax.set(xlabel=r"$\theta$", ylabel="density", title=r"NPE posterior at $x_{obs}$")
ax.legend(fontsize=8)
plt.show()

## All three methods side by side

Below, rejection ABC (Exercise 1) and the MDN (Exercise 2) are rerun from their solutions. The MDN is trained on the same 5,000 simulations as NPE.

In [7]:
# ---------------- Rejection ABC, from Exercise 1 ----------------
def rejection_abc(x_obs, epsilon, n_simulations):
    theta = prior.sample((n_simulations,)).squeeze(-1)
    x_sim = simulate(theta)
    return theta[torch.abs(x_sim - x_obs) < epsilon].numpy()

EPSILON, N_SIMULATIONS = 0.15, 100_000
abc_posterior_samples = rejection_abc(x_obs, EPSILON, N_SIMULATIONS)

# ---------------- MDN, from Exercise 2 ----------------
class MDN(nn.Module):
    def __init__(self, n_components=6, n_hidden=64, x_scale=10.0):
        super().__init__()
        self.x_scale = x_scale
        self.mlp = nn.Sequential(nn.Linear(1, n_hidden), nn.Tanh(), nn.Linear(n_hidden, n_hidden), nn.Tanh())
        self.logits_head = nn.Linear(n_hidden, n_components)
        self.mu_head = nn.Linear(n_hidden, n_components)
        self.log_sigma_head = nn.Linear(n_hidden, n_components)

    def forward(self, x):
        x_scaled = x / self.x_scale
        h = self.mlp(x_scaled)
        return self.logits_head(h), self.mu_head(h), self.log_sigma_head(h)

def mixture_log_density(model, theta, x):
    pi_logits, mu, log_sigma = model(x)
    log_normal = -0.5 * ((theta - mu) / torch.exp(log_sigma)) ** 2 - log_sigma - 0.5 * math.log(2 * math.pi)
    return torch.logsumexp(torch.log_softmax(pi_logits, dim=-1) + log_normal, dim=-1)

torch.manual_seed(SEED)
mdn = MDN(n_components=6)
optimizer = torch.optim.Adam(mdn.parameters(), lr=3e-3)
for step in range(3000):
    optimizer.zero_grad()
    loss = -mixture_log_density(mdn, theta_train, x_train).mean()
    loss.backward()
    optimizer.step()
print(f"MDN final training NLL: {loss.item():.3f}")

theta_grid = np.linspace(THETA_LOW, THETA_HIGH, 1000)
with torch.no_grad():
    theta_grid_t = torch.tensor(theta_grid, dtype=torch.float32)[:, None]
    mdn_posterior_density = mixture_log_density(mdn, theta_grid_t, x_obs_t).exp().numpy()

MDN final training NLL: 1.687


In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.hist(abc_posterior_samples, bins=60, range=(THETA_LOW, THETA_HIGH), density=True, histtype="stepfilled",
        color=COLORS["abc"], alpha=0.35, label=rf"Rejection ABC ($\epsilon$ = {EPSILON}, {N_SIMULATIONS:,} simulations)")
ax.plot(theta_grid, mdn_posterior_density, color=COLORS["mdn"], lw=2.5, label=f"MDN ({N_TRAIN:,} simulations)")
ax.hist(npe_posterior_samples, bins=80, range=(THETA_LOW, THETA_HIGH), density=True, histtype="step",
        color=COLORS["npe"], lw=2, label=f"NPE, neural spline flow ({N_TRAIN:,} simulations)")
if SHOW_GROUND_TRUTH:
    ax.plot(*grid_posterior(x_obs), color=COLORS["truth"], lw=1.5, ls="--", label="true posterior (grid)")
ax.axvline(THETA_TRUE, color="k", ls="--", lw=1, label=r"$\theta_{true}$")
ax.set(xlabel=r"$\theta$", ylabel="density", title=r"Three approximations of $p(\theta \mid x_{obs})$")
ax.legend(fontsize=8)
plt.show()

## A few knobs worth knowing

We used defaults for almost everything. The settings most worth knowing about:

- **Density estimator.** `density_estimator` accepts `"maf"` (masked autoregressive flow, the default), `"nsf"` (neural spline flow) and `"mdn"` (a mixture density network, like Exercise 2). For more control, `sbi.neural_nets.posterior_nn` builds an estimator with custom settings, such as `hidden_features`, `num_transforms` (flows) or `num_components` (MDN).
- **Training.** `inference.train()` takes, among others, `learning_rate` (default `5e-4`), `training_batch_size` (`200`), `validation_fraction` (`0.1`), `stop_after_epochs` (`20`: stop once the validation loss hasn't improved for this many epochs) and `max_num_epochs`.

For example, here is `sbi`'s own MDN with 10 components, as in Exercise 2:

In [ ]:
from sbi.neural_nets import posterior_nn

torch.manual_seed(SEED)
mdn_estimator = posterior_nn(model="mdn", num_components=10, hidden_features=50)
inference_mdn = NPE(prior=prior, density_estimator=mdn_estimator)
inference_mdn.append_simulations(theta_train, x_train)
inference_mdn.train(learning_rate=1e-3, stop_after_epochs=20)
sbi_mdn_samples = inference_mdn.build_posterior().sample((10_000,), x=x_obs_t).numpy().ravel()

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(npe_posterior_samples, bins=80, range=(THETA_LOW, THETA_HIGH), density=True, histtype="step",
        color=COLORS["npe"], lw=2, label='NPE, density_estimator="nsf"')
ax.hist(sbi_mdn_samples, bins=80, range=(THETA_LOW, THETA_HIGH), density=True, histtype="step",
        color=COLORS["mdn"], lw=2, label="NPE, posterior_nn(model=\"mdn\")")
if SHOW_GROUND_TRUTH:
    ax.plot(*grid_posterior(x_obs), color=COLORS["truth"], lw=1.5, ls="--", label="true posterior (grid)")
ax.set(xlabel=r"$\theta$", ylabel="density")
ax.legend(fontsize=8)
plt.show()

## ABC vs. NPE on the same simulation budget

Simulations are usually the expensive part. How do ABC and NPE compare when both get the same number of simulations $N$?

- **ABC** spends all $N$ simulations on this one observation and keeps the fraction that land within $\epsilon$.
- **NPE** uses the same $N$ simulations to train $q_\phi(\theta \mid x)$, which then works for any observation.

We measure accuracy as the total variation (TV) distance to the grid posterior, $\mathrm{TV}(p, q) = \tfrac12 \int |p(\theta) - q(\theta)|\,d\theta$. It is 0 for identical distributions and 1 for distributions that don't overlap at all. We estimate it by binning samples from each method, and from the grid posterior, into the same histogram bins. (This uses the grid posterior whatever `SHOW_GROUND_TRUTH` is set to.)

**Exercise 3c.** Inside the loop, train NPE on the `n` simulations `(theta_n, x_n)` and draw 10,000 posterior samples at `x_obs_t`, the same steps as Exercises 3a and 3b. The largest budget takes a minute or two to train.

In [ ]:
BUDGETS = [500, 2_000, 10_000]
bin_edges = np.linspace(THETA_LOW, THETA_HIGH, 43)

def sample_grid_posterior(x, n):
    """Draw samples from the grid posterior by inverse-CDF sampling."""
    theta_grid, density = grid_posterior(x)
    cdf = np.cumsum(density)
    return np.interp(torch.rand(n).numpy(), cdf / cdf[-1], theta_grid)

def tv_distance(samples_a, samples_b):
    p = np.histogram(samples_a, bins=bin_edges)[0] / len(samples_a)
    q = np.histogram(samples_b, bins=bin_edges)[0] / len(samples_b)
    return 0.5 * np.abs(p - q).sum()

reference_samples = sample_grid_posterior(x_obs, 100_000)
tv = {"abc": [], "npe": []}

fig, axes = plt.subplots(1, len(BUDGETS) + 1, figsize=(17, 3.6))
for ax, n in zip(axes, BUDGETS):
    abc_n = rejection_abc(x_obs, EPSILON, n)

    torch.manual_seed(SEED)
    theta_n = prior.sample((n,))
    x_n = simulate(theta_n)

    # EXERCISE 3c: train NPE ("nsf") on (theta_n, x_n), and store 10_000 posterior samples at x_obs_t
    # as a 1D numpy array called `npe_n`. Pass show_progress_bars=False to NPE(...) and .sample(...) to keep the output short.
    inference_n = ...
    ...  # inference_n.append_simulations(theta_n, x_n)
    ...  # inference_n.train()
    npe_n = ...

    tv["abc"].append(tv_distance(abc_n, reference_samples) if len(abc_n) else 1.0)
    tv["npe"].append(tv_distance(npe_n, reference_samples))

    ax.hist(npe_n, bins=bin_edges, density=True, color=COLORS["npe"], alpha=0.5, label="NPE")
    if len(abc_n):
        ax.hist(abc_n, bins=bin_edges, density=True, color=COLORS["abc"], alpha=0.5, label=f"ABC ({len(abc_n)} accepted)")
    if SHOW_GROUND_TRUTH:
        ax.plot(*grid_posterior(x_obs), color=COLORS["truth"], lw=1.2)
    ax.axvline(THETA_TRUE, color="k", ls="--", lw=1)
    ax.set(xlabel=r"$\theta$", title=f"N = {n:,} simulations")
    ax.legend(fontsize=7)

axes[-1].plot(BUDGETS, tv["abc"], "o-", color=COLORS["abc"], label="ABC")
axes[-1].plot(BUDGETS, tv["npe"], "o-", color=COLORS["npe"], label="NPE")
axes[-1].set(xscale="log", ylim=(0, 1), xlabel="simulations N", ylabel="TV distance to true posterior")
axes[-1].legend()
plt.tight_layout()
plt.show()

## Is the posterior calibrated? Simulation-based calibration

Comparing against a grid posterior is a luxury of toy problems. In practice, NPE is checked with **simulation-based calibration (SBC)**, a standard part of the SBI workflow:

1. Draw $\theta^* \sim p(\theta)$ and simulate $x^* \sim p(x \mid \theta^*)$.
2. Sample from the estimated posterior $q_\phi(\theta \mid x^*)$, and record the *rank* of $\theta^*$ among those samples.
3. Repeat many times.

If the posterior is calibrated, $\theta^*$ is just another draw from it, so its rank is uniformly distributed. A ∪-shaped rank histogram means the posterior is too narrow (overconfident), a ∩-shaped one means it is too wide, and a slope means it is biased.

SBC checks calibration *averaged over the prior*, not accuracy at one particular $x_{obs}$. The prior itself passes SBC perfectly, even though it ignores the data.

In [ ]:
# Optional: SBC for the NPE posterior trained above (takes under a minute)
from sbi.analysis.plot import sbc_rank_plot
from sbi.diagnostics import run_sbc

N_SBC, N_POSTERIOR_SAMPLES = 300, 500

torch.manual_seed(SEED)
theta_sbc = prior.sample((N_SBC,))
x_sbc = simulate(theta_sbc)
ranks, dap_samples = run_sbc(theta_sbc, x_sbc, posterior, num_posterior_samples=N_POSTERIOR_SAMPLES)

fig, ax = sbc_rank_plot(ranks, num_posterior_samples=N_POSTERIOR_SAMPLES, plot_type="cdf")
plt.show()

## When things go wrong

A few failure modes worth trying yourself:

- **A prior that is too narrow.** Train NPE with a prior that excludes $\theta_{true}$, e.g. `BoxUniform(low=torch.tensor([2.0]), high=torch.tensor([10.5]))`, and query it at our $x_{obs}$. The network has never seen a simulation like $x_{obs}$, and `sbi` only returns samples inside the prior, so the result can look confident and still be wrong. SBC run with that same narrow prior won't catch this: it only tests calibration within the prior.
- **An observation unlike the training data.** Even with the right prior, an $x_{obs}$ far outside the range of the simulations (try `x = 40`) forces the network to extrapolate. Compare its posterior with the grid posterior.
- **Too few simulations, or an undertrained network.** You saw this in the budget comparison. Without a ground truth, SBC and posterior predictive checks (simulating from posterior samples and comparing to $x_{obs}$) are the tools that catch it.

## Using NPE on your own problem

The whole pipeline in one function. Swap in your own `simulator` (a function from a batch of parameters to a batch of observations) and prior.

In [12]:
def run_npe(simulator, prior, x_observed, n_simulations=5_000, n_samples=10_000, density_estimator="nsf"):
    """Simulate -> train -> sample. `simulator` maps a (n, d_theta) float32 tensor to a (n, d_x) float32 tensor."""
    theta = prior.sample((n_simulations,))
    x = simulator(theta)
    inference = NPE(prior=prior, density_estimator=density_estimator)
    inference.append_simulations(theta, x)
    inference.train()
    posterior = inference.build_posterior()
    return posterior, posterior.sample((n_samples,), x=x_observed)

# Example (not run):
# my_prior = BoxUniform(low=torch.tensor([0.0, -1.0]), high=torch.tensor([1.0, 1.0]))
# def my_simulator(theta):
#     ...  # return a (n, d_x) float32 tensor
# my_posterior, my_samples = run_npe(my_simulator, my_prior, x_observed=torch.tensor([[...]]))

## Questions to think about

1. NPE and the MDN are both conditional density estimators trained by maximum likelihood. What differs between them here, and does the difference show in the posteriors?
2. As problems get harder (more parameters, higher-dimensional or structured data such as time series), where would you expect NPE to pull ahead of the hand-written MDN?
3. What does `sbi`'s convenience cost you compared with Exercise 2's from-scratch version? Think about what you can see and what you can control.
4. In the budget comparison, from what $N$ does NPE beat ABC? How would the answer change if you had 100 observations instead of one?